# Display D-vs-beta Heatmap Results

This notebook reads a fixed-c success matrix, uses the saved `D` and `beta` axes, and overlays the theoretical `D_min` curve. The preset `c` is a uniform experimental threshold, not an estimate of the actual contraction rate.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

## Parameters

These values must match the parameters used to generate the heatmap files.

In [ ]:
n = 100
q = 0.8
beta_min = 0.00
beta_max = 0.02
D_min = 1
D_max = 30
c = 1e-2
num_samples = 100
T_max = 20_000
corruption_type = "sup_c"
# corruption_type = "adversarial"

## Locate Files

In [ ]:
working_dir = Path.cwd().resolve()
repo_root = next(
    (path for path in (working_dir, *working_dir.parents)
     if (path / "heat_map_raw_data").is_dir()),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        f"Could not find heat_map_raw_data from {working_dir} or its parents."
    )

data_dir = repo_root / "heat_map_raw_data"
base_suffix = (
    f"__n={n}"
    f"__q={q * 100:2.0f}"
    f"__beta_min={beta_min * 100:.0f}"
    f"__beta_max={beta_max * 100:.0f}"
    f"__D_min={D_min}"
    f"__D_max={D_max}"
    f"__c={c:1.0e}"
    f"__num_samples={num_samples}"
    f"__T_max={T_max}"
    f"__corruption_type={corruption_type}"
)

success_path = data_dir / f"D_vs_beta{base_suffix}.txt"
d_min_path = data_dir / f"D_vs_beta__D_min{base_suffix}.txt"
D_samples_path = data_dir / f"D_vs_beta__D_samples{base_suffix}.txt"
beta_samples_path = data_dir / f"D_vs_beta__beta_samples{base_suffix}.txt"

paths = (success_path, d_min_path, D_samples_path, beta_samples_path)
for path in paths:
    print(path)
    if not path.exists():
        raise FileNotFoundError(path)

If the parameters do not match an existing result, list the available D-vs-beta files.

In [ ]:
for path in sorted(data_dir.glob("D_vs_beta*.txt")):
    print(path.name)

## Load and Validate Data

In [ ]:
D_values = np.ravel(np.loadtxt(D_samples_path))
beta_values = np.ravel(np.loadtxt(beta_samples_path))
d_min_values = np.ravel(np.loadtxt(d_min_path))
success = np.atleast_2d(np.loadtxt(success_path))

expected_shape = (len(D_values), len(beta_values))
if success.shape == expected_shape:
    pass
elif success.T.shape == expected_shape:
    print(f"Transposing success matrix from {success.shape} to {expected_shape}.")
    success = success.T
else:
    raise ValueError(
        f"Success matrix has shape {success.shape}; expected D x beta = {expected_shape}."
    )

if len(d_min_values) != len(beta_values):
    raise ValueError(
        f"D_min has length {len(d_min_values)}; expected {len(beta_values)}."
    )

print("success shape:", success.shape)
print("D range:", D_values[0], "to", D_values[-1])
print("beta range:", beta_values[0], "to", beta_values[-1])

## Success Heatmap

In [ ]:
def cell_edges(values):
    values = np.asarray(values, dtype=float)
    if len(values) == 1:
        return np.array([values[0] - 0.5, values[0] + 0.5])
    midpoints = (values[:-1] + values[1:]) / 2
    return np.r_[
        values[0] - (midpoints[0] - values[0]),
        midpoints,
        values[-1] + (values[-1] - midpoints[-1]),
    ]

beta_edges = cell_edges(beta_values)
D_edges = cell_edges(D_values)

fig, ax = plt.subplots(figsize=(10, 6))
image = ax.pcolormesh(
    beta_edges,
    D_edges,
    success,
    shading="flat",
    vmin=0,
    vmax=1,
    cmap="jet",
)
ax.plot(
    beta_values,
    d_min_values,
    linewidth=3,
    color="black",
    label="theoretical D_min",
)
fig.colorbar(image, ax=ax, label="success rate")
ax.set_title(f"Success heatmap ({corruption_type}, fixed c = {c:g})")
ax.set_xlabel("beta")
ax.set_ylabel("D")
ax.legend()
fig.tight_layout()

## Quick Numeric Summaries

In [ ]:
print("success min/mean/max:", np.nanmin(success), np.nanmean(success), np.nanmax(success))
print("D_min min/max:", np.nanmin(d_min_values), np.nanmax(d_min_values))